In [14]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from torchvision.datasets import ImageFolder

# =====================================
# 1. Device
# =====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================
# 2. Dataset paths (used to get class names)
# =====================================
dress_type_train_path = "model/dataset-splitted/train"
color_train_path = "model/dataset-splitted/train"

# Load datasets just to get classes
train_data = ImageFolder(dress_type_train_path)
dress_type_classes = train_data.classes

color_train_data = ImageFolder(color_train_path)
color_classes = color_train_data.classes

# =====================================
# 3. Model paths
# =====================================
model_type_path = r"model/final-model/dress-type-model-20251019-013406-1.pth"
model_color_path = r"model/final-model/dress-type-model-20251018-214332.pth"

# =====================================
# 4. Create architectures and load weights
# =====================================
# Dress Type Model
model_type = models.resnet50(weights=None)
model_type.fc = nn.Linear(2048, len(dress_type_classes))
state_dict_type = torch.load(model_type_path, map_location=device)
model_type.load_state_dict(state_dict_type, strict=False)
model_type.to(device)
model_type.eval()

# Color/Pattern Model
model_color = models.resnet50(weights=None)
model_color.fc = nn.Linear(2048, len(color_classes))
state_dict_color = torch.load(model_color_path, map_location=device)
model_color.load_state_dict(state_dict_color, strict=False)
model_color.to(device)
model_color.eval()

# =====================================
# 5. Preprocessing
# =====================================
preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# =====================================
# 6. Combined prediction function
# =====================================
def predict_dress(image_path):
    image = Image.open(image_path).convert("RGB")
    img_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        # Dress type prediction
        type_logits = model_type(img_tensor)
        type_idx = type_logits.argmax(1).item()
        type_label = dress_type_classes[type_idx]

        # Color/pattern prediction
        color_logits = model_color(img_tensor)
        color_idx = color_logits.argmax(1).item()
        color_label = color_classes[color_idx]

    return f"color: {color_label}, type: {type_label}"

# =====================================
# 7. Example usage
# =====================================
result = predict_dress("model/abstract-buttoned-top-test.png")
print("🧾 Final Prediction:", result)

🧾 Final Prediction: color: Abstract_Buttoned_Top, type: Abstract_Print_Draped_Blazer
